# Concurrency & Recovery Demos

Maps Neo4j to Elmasri & Navathe ch. 21–22. Runs against the 10-patient subset from `load_subset.py`.

**Prerequisites**
- `docker compose up -d`
- `python load_subset.py`

**Demo isolation.** Writes only touch demo-scoped properties (`p.demo_*`) on real PATIENT nodes; ephemeral entities carry `:DemoNode`. The setup cell wipes prior demo state, so the notebook is idempotent.

Each cell prints a slice of `query.log` or `debug.log` at the end — no terminal-tailing needed.

| # | Theme       | Demo                              |
|---|-------------|-----------------------------------|
| 1 | Transaction | Atomic multi-entity rollback      |
| 2 | Transaction | Checkpoint + WAL truncation       |
| 3 | Transaction | Crash recovery via WAL replay     |
| 4 | Concurrency | Non-repeatable read               |
| 5 | Concurrency | Lost update (naive vs atomic SET) |


In [1]:
import os, time, uuid, subprocess
from threading import Event, Thread, Barrier

import neo4j
from neo4j.exceptions import ConstraintError
from dotenv import load_dotenv

load_dotenv()
URI       = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
USER      = os.getenv("NEO4J_USERNAME", "neo4j")
PWD       = os.getenv("NEO4J_PASSWORD", "password123")
DB        = os.getenv("NEO4J_DATABASE", "neo4j")
CONTAINER = "neo4j-demo"
RUN_ID    = uuid.uuid4().hex[:8]


def session():
    return driver.session(database=DB)


def show_log(logfile, n=20, grep=None):
    cmd = (f"grep -iE '{grep}' /logs/{logfile} 2>/dev/null | tail -n {n}"
           if grep else f"tail -n {n} /logs/{logfile}")
    out = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c", cmd],
        capture_output=True, text=True, timeout=15,
    ).stdout or "(no matches)"
    print(f"--- /logs/{logfile}  ({grep or 'tail'}) ---\n{out}")


def cleanup():
    with session() as s:
        s.run("MATCH (n:DemoNode) DETACH DELETE n").consume()
        s.run("MATCH (p:PATIENT) "
              "REMOVE p.demo_note, p.demo_last_visit, "
              "       p.demo_visit_count, p.demo_run_id").consume()


driver = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD))
driver.verify_connectivity()
cleanup()

with session() as s:
    row = s.run("""
        MATCH (p:PATIENT)-[:HAS_IMAGE]->(i:IMAGE)
        WHERE NOT p:DemoNode
        RETURN p.patient_id AS pid, i.instance_uid AS uid LIMIT 1
    """).single()
if not row:
    raise RuntimeError("No PATIENT/IMAGE found. Run `python load_subset.py`.")

PID          = row["pid"]
EXISTING_UID = row["uid"]
print(f"RUN_ID={RUN_ID}  PID={PID}")
print(f"sample instance_uid: {EXISTING_UID}")

RUN_ID=8e72ff69  PID=27
sample instance_uid: 1.3.6.1.4.1.9590.100.1.2.59620512812470186337816449881316634272


## Demo 1 — Atomic multi-entity transaction (commit vs rollback)

ACID atomicity: a transaction is all-or-nothing.

The cell runs the **same 4-step workflow twice**:

1. update `PATIENT.demo_last_visit`
2. create an `:Annotation:DemoNode`
3. link it to the PATIENT and one of its real IMAGEs
4. create an `:IMAGE`

**Round A — success.** Step 4 uses a *fresh* `instance_uid`. All four writes commit; the state changes.

**Round B — failure.** Step 4 reuses an *existing* `instance_uid`, violating the `UNIQUE` constraint. The transaction rolls back automatically. The state after Round B must equal the state after Round A — proving the failed transaction left no trace.

In [2]:
def state():
    with session() as s:
        r = s.run("""
            MATCH (p:PATIENT {patient_id: $pid})
            OPTIONAL MATCH (a:Annotation:DemoNode)
            RETURN toString(p.demo_last_visit) AS visit, count(a) AS annots
        """, pid=PID).single()
    return r["visit"], r["annots"]


def workflow(uid, new_visit):
    aid = f"{RUN_ID}-{uid[:6]}"
    with session() as s, s.begin_transaction() as tx:
        tx.run("""
            MATCH (p:PATIENT {patient_id: $pid})-[:HAS_IMAGE]->(i:IMAGE)
            WITH p, i LIMIT 1
            SET p.demo_last_visit = datetime($visit)
            CREATE (a:Annotation:DemoNode {annotation_id: $aid})
            MERGE (a)-[:ANNOTATES]->(p)
            MERGE (a)-[:ABOUT_IMAGE]->(i)
        """, pid=PID, visit=new_visit, aid=aid)
        tx.run("CREATE (:IMAGE:DemoNode {instance_uid: $uid})", uid=uid)


with session() as s:
    s.run("MATCH (p:PATIENT {patient_id: $pid}) "
          "SET p.demo_last_visit = datetime('2025-01-01')", pid=PID).consume()
initial = state()
print(f"INITIAL      : {initial}")

# Round A: fresh UID -> commits
workflow(f"{RUN_ID}-fresh-uid", "2026-05-14")
after_ok = state()
print(f"AFTER ROUND A: {after_ok}   ← committed")
assert after_ok != initial

# Round B: duplicate UID -> ConstraintError -> auto-rollback
try:
    workflow(EXISTING_UID, "2027-01-01")
except ConstraintError as e:
    print(f"\nConstraintError -> auto-rollback")
after_fail = state()
print(f"AFTER ROUND B: {after_fail}   ← rolled back (same as Round A)")
assert after_fail == after_ok

show_log("query.log", grep="Annotation|demo_last_visit")

INITIAL      : ('2025-01-01T00:00:00Z', 0)
AFTER ROUND A: ('2026-05-14T00:00:00Z', 1)   ← committed

ConstraintError -> auto-rollback
AFTER ROUND B: ('2026-05-14T00:00:00Z', 1)   ← rolled back (same as Round A)
--- /logs/query.log  (Annotation|demo_last_visit) ---
(no matches)


## Demo 2 — Checkpoint + WAL truncation

The WAL grows on every commit. A checkpoint flushes dirty pages, writes a checkpoint marker, and lets older WAL segments be pruned.

This bounds recovery time: Demo 3 only needs to replay from the *last checkpoint*, not from db creation.

The container is configured (`docker-compose.yml`) to checkpoint every **2 log chunks** (≈ 40 transactions in this workload) or every **2 seconds**. The cell counts `"Checkpoint started"` lines in `debug.log` before and after 200 writes — the **delta** is the demo's artifact and should be ≥ 5.

*Note:* `db.checkpoint.interval.tx` in Neo4j 5 counts *log chunks*, not raw transactions (the trigger reason in the log says `"every N log chunks threshold"`). `CALL db.checkpoint()` is enterprise-only — we rely on the auto-trigger.

In [3]:
def n_checkpoints():
    p = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c",
         "grep -ci 'checkpoint started' /logs/debug.log || true"],
        capture_output=True, text=True,
    )
    return int(p.stdout.strip() or 0)


# Diagnostic: which checkpoint thresholds is the live container using?
with session() as s:
    cfg = s.run(
        "CALL dbms.listConfig() YIELD name, value "
        "WHERE name CONTAINS 'checkpoint.interval' RETURN name, value"
    ).data()
print("Active checkpoint config:")
for r in cfg:
    print(f"  {r['name']:35} = {r['value']}")

before = n_checkpoints()
print(f"\nBEFORE: {before} 'Checkpoint started' lines in debug.log")

N = 300
t0 = time.perf_counter()
with session() as s:
    for i in range(N):
        s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_note = $v",
              pid=PID, v=f"tick-{i}").consume()
elapsed = time.perf_counter() - t0
print(f"\n{N} writes done in {elapsed:.1f}s.")

time.sleep(3)  # catch the trailing time-based checkpoints

after = n_checkpoints()
print(f"\nAFTER : {after} 'Checkpoint started' lines in debug.log")
print(f"DELTA : {after - before} checkpoint(s) fired during this cell.\n")

show_log("debug.log", grep="checkpoint started|prun")

Active checkpoint config:
  db.checkpoint.interval.time         = 1s
  db.checkpoint.interval.tx           = 1
  db.checkpoint.interval.volume       = 250.00MiB

BEFORE: 17 'Checkpoint started' lines in debug.log

300 writes done in 2.4s.

AFTER : 21 'Checkpoint started' lines in debug.log
DELTA : 4 checkpoint(s) fired during this cell.

--- /logs/debug.log  (checkpoint started|prun) ---
2026-05-14 15:48:53.669+0000 INFO  [o.n.k.i.t.l.c.CheckPointerImpl] [neo4j/845e4053] Checkpoint triggered by "Scheduled checkpoint for every 2 log chunks threshold" @ txId: 264, append index: 264 checkpoint started...
2026-05-14 15:48:53.782+0000 INFO  [o.n.k.i.t.l.p.LogPruningImpl] [neo4j/845e4053] No log version pruned. The strategy used was '2 days 2147483648 size'. 
2026-05-14 15:48:55.783+0000 INFO  [o.n.k.i.t.l.c.CheckPointerImpl] [neo4j/845e4053] Checkpoint triggered by "Scheduled checkpoint for every 2 log chunks threshold" @ txId: 346, append index: 346 checkpoint started...
2026-05-14 15:48:5

## Demo 3 — Crash recovery via WAL replay (committed vs uncommitted)

Durability + crash atomicity in one demo:

- **Committed transaction → survives.** A `:CrashMarker {status: 'committed'}` is written and committed before the kill. The WAL has the COMMIT record; recovery replays it.
- **Uncommitted transaction → vanishes.** A second `:CrashMarker {status: 'uncommitted'}` is written inside an open transaction that we deliberately *do not commit*. Then we SIGKILL. The WAL has the data record but no COMMIT — recovery discards it.

After the container restarts, we query for both markers. The committed one must be there; the uncommitted one must be gone.

This is the demo Aura cannot run.

In [4]:
committed_id   = f"{RUN_ID}-COMMITTED"
uncommitted_id = f"{RUN_ID}-UNCOMMITTED"

# 1. Auto-commit marker (durability test target).
with session() as s:
    s.run("""
        CREATE (:CrashMarker:DemoNode {
            marker_id: $m, status: 'committed', written_at: datetime()
        })
    """, m=committed_id).consume()
print(f"COMMITTED   : {committed_id}  ← should survive")

# 2. Open a tx, write, but DO NOT commit (crash-atomicity test target).
limbo_session = driver.session(database=DB)
limbo_tx = limbo_session.begin_transaction()
limbo_tx.run("""
    CREATE (:CrashMarker:DemoNode {
        marker_id: $m, status: 'uncommitted', written_at: datetime()
    })
""", m=uncommitted_id)
print(f"UNCOMMITTED : {uncommitted_id}  ← should NOT survive (tx still open)")

# 3. SIGKILL while limbo_tx is mid-flight (no chance for client rollback).
subprocess.run(["docker", "kill", "-s", "SIGKILL", CONTAINER],
               check=True, capture_output=True)
print("SIGKILL sent")
for closer in (limbo_tx, limbo_session, driver):
    try: closer.close()
    except Exception: pass

# 4. Restart container, reconnect (fast poll: 0.5s interval, 2s connect timeout).
subprocess.run(["docker", "start", CONTAINER], check=True, capture_output=True)
print("Container restarted; waiting for Bolt...", end="", flush=True)

t0 = time.perf_counter()
driver = None
for _ in range(60):
    try:
        driver = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD), connection_timeout=2)
        driver.verify_connectivity()
        break
    except Exception:
        if driver: driver.close()
        driver = None
        print(".", end="", flush=True)
        time.sleep(0.5)
else:
    raise RuntimeError("Bolt didn't come back in 60 polls")
print(f" up in {time.perf_counter() - t0:.1f}s\n")

show_log("debug.log", n=30, grep="recover|replay")

# 5. Verify: committed marker present, uncommitted marker absent.
with session() as s:
    found_c = s.run("MATCH (m:CrashMarker {marker_id: $m}) RETURN m.marker_id AS id",
                    m=committed_id).single()
    found_u = s.run("MATCH (m:CrashMarker {marker_id: $m}) RETURN m.marker_id AS id",
                    m=uncommitted_id).single()

print(f"\nCommitted marker present?   {found_c is not None}   ({found_c})")
print(f"Uncommitted marker present? {found_u is not None}   ({found_u})")
assert found_c is not None, "Durability violated: committed write was lost"
assert found_u is None,     "Crash atomicity violated: uncommitted write persisted"
print("\n✓ committed write durable; uncommitted write rolled back by recovery")

COMMITTED   : 8e72ff69-COMMITTED  ← should survive
UNCOMMITTED : 8e72ff69-UNCOMMITTED  ← should NOT survive (tx still open)


[#CBA6]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0))): ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None)


SIGKILL sent
Container restarted; waiting for Bolt..................... up in 9.9s

--- /logs/debug.log  (recover|replay) ---
2026-05-14 15:48:46.041+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  20% completed
2026-05-14 15:48:46.041+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  30% completed
2026-05-14 15:48:46.041+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  40% completed
2026-05-14 15:48:46.041+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  50% completed
2026-05-14 15:48:46.590+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  60% completed
2026-05-14 15:48:46.591+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  70% completed
2026-05-14 15:48:46.592+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  80% completed
2026-05-14 15:48:46.623+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  90% completed
2026-05-14 15:48:46.624+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053] 100% completed
2026-05-14 15:48:47.292+0000 INFO  [o.n.k.d.Database] [neo4j/845e4053] Recovery in 'full' mode compl

## Demo 4 — Non-repeatable read

Neo4j defaults to READ COMMITTED. That prevents *dirty* reads but allows *non-repeatable* ones: two reads of the same value in one transaction can differ if another transaction commits between them.

```
READER tx:  read #1 -------------- read #2 -- commit
WRITER tx:           SET -- commit
```

The two threads synchronise via `Event`s, so the ordering — and therefore the outcome — is deterministic.

In [5]:
with session() as s:
    s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_note = 'initial'",
          pid=PID).consume()

phase1, phase2 = Event(), Event()
reads = {}


def reader():
    with session() as s, s.begin_transaction() as tx:
        reads["r1"] = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                             "RETURN p.demo_note AS n", pid=PID).single()["n"]
        print(f"reader read#1 = {reads['r1']!r}")
        phase1.set()
        phase2.wait(timeout=10)
        reads["r2"] = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                             "RETURN p.demo_note AS n", pid=PID).single()["n"]
        print(f"reader read#2 = {reads['r2']!r}   (same tx, after writer commit)")


def writer():
    phase1.wait()
    with session() as s:
        s.run("MATCH (p:PATIENT {patient_id: $pid}) "
              "SET p.demo_note = 'mutated'", pid=PID).consume()
    print("writer committed")
    phase2.set()


a, b = Thread(target=reader), Thread(target=writer)
a.start(); b.start(); a.join(); b.join()

assert reads["r1"] != reads["r2"], "non-repeatable read did not occur"
print(f"\n✓ non-repeatable read: r1={reads['r1']!r}  r2={reads['r2']!r}\n")

show_log("query.log", grep="demo_note")

reader read#1 = 'initial'
writer committed
reader read#2 = 'mutated'   (same tx, after writer commit)

✓ non-repeatable read: r1='initial'  r2='mutated'

--- /logs/query.log  (demo_note) ---
(no matches)


## Demo 5 — Lost update: naive RMW vs atomic SET

When application code does its own read-modify-write across statements, the read takes no lock. Two threads can both read `v`, both write `v+1`. One increment vanishes silently.

Both threads use a `Barrier` to read in lock-step, so the race is **deterministic**:

| Round  | Pattern                          | Expected | Actual                              |
|--------|----------------------------------|----------|-------------------------------------|
| Naive  | `v = SELECT; SET = v+1`          | 200      | exactly **100** (every iteration both reads see the same `v`) |
| Atomic | `SET p.x = p.x + 1`              | 200      | exactly **200**                     |

Retries can't save you — lost updates raise no error.

In [6]:
N = 100


def naive_thread(barrier):
    with session() as s:
        for _ in range(N):
            with s.begin_transaction() as tx:
                v = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                           "RETURN coalesce(p.demo_visit_count, 0) AS v",
                           pid=PID).single()["v"]
                barrier.wait()
                tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                       "SET p.demo_visit_count = $v", pid=PID, v=v + 1)


def atomic_thread():
    with session() as s:
        for _ in range(N):
            s.run("MATCH (p:PATIENT {patient_id: $pid}) "
                  "SET p.demo_visit_count = coalesce(p.demo_visit_count, 0) + 1",
                  pid=PID).consume()


def reset_count():
    with session() as s:
        s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_visit_count = 0",
              pid=PID).consume()


def get_count():
    with session() as s:
        return s.run("MATCH (p:PATIENT {patient_id: $pid}) "
                     "RETURN p.demo_visit_count AS v", pid=PID).single()["v"]


def run_pair(target, *args):
    ts = [Thread(target=target, args=args) for _ in range(2)]
    for t in ts: t.start()
    for t in ts: t.join()


reset_count()
print(f"NAIVE (Barrier-locked), 2 threads x {N}:")
run_pair(naive_thread, Barrier(2))
nv = get_count()
print(f"  -> {nv}   (expected {N}, lost {2*N - nv})\n")
assert nv == N

reset_count()
print(f"ATOMIC SET, 2 threads x {N}:")
run_pair(atomic_thread)
av = get_count()
print(f"  -> {av}   (expected {2*N})\n")
assert av == 2 * N

show_log("query.log", grep="demo_visit_count")
cleanup()
print("\nDemo state cleaned.")

NAIVE (Barrier-locked), 2 threads x 100:
  -> 100   (expected 100, lost 100)

ATOMIC SET, 2 threads x 100:
  -> 200   (expected 200)

--- /logs/query.log  (demo_visit_count) ---
(no matches)

Demo state cleaned.
